# Patel, Kim et al. (2026) from dandiset 001454: patch clamp

Paper. Patel RR, Kim KN, Patarino M, et al. (2026). Social isolation recruits
amygdala–medial prefrontal cortex projections to escalate alcohol drinking in male mice.
*Nature Neuroscience*. [doi:10.1038/s41593-026-02413-x](https://doi.org/10.1038/s41593-026-02413-x).
The claims file was written from the version of record (PMC13633117), its figure legends,
Extended Data legends, Methods and reporting summary; Supplementary Tables 1 and 2 were not in the
text read.

Dandiset. [DANDI:001454](https://dandiarchive.org/dandiset/001454/0.260911.2319), version
0.260911.2319. The paper cites the dandiset without a version ("Unprocessed data are available via
the DANDI Archive at https://dandiarchive.org/dandiset/001454"). This is the only published
version, and its release notes say it holds "all recordings and imaging data used in the Nature
Neuroscience publication". The draft has the same 359 assets and size; it is not pinned because it
can change.

This notebook tests the patch-clamp claims (Figure 2, Extended Data Figs. 2 and 3, Supplementary
Table 1). The imaging claims are in `imaging_patel_2026.ipynb`. No random numbers are drawn here.

| Claim | Status |
|---|---|
| fig2c-male-blampfc-excitability-up | tested |
| fig2d-female-blampfc-excitability-down | blocked: 11 of 14 isolated female files hold no recording |
| fig2e-female-higher-baseline-excitability | tested |
| fig2-threeway-sex-current-housing | blocked: needs the isolated females |
| supt1-male-input-resistance-up | tested |
| supt1-female-input-resistance-threshold | blocked: needs the isolated females |
| ed2e-nonspecific-bla-excitability | tested |
| ed2bcd-nonspecific-bla-no-rank | tested |
| ed3c-male-rank-blampfc-excitability | tested |
| ed3d-female-rank-blampfc-none | tested |
| Imaging claims (Figs. 3, 5, 6; Extended Data Figs. 5, 6, 8) | in `imaging_patel_2026.ipynb` |
| Home-cage 2BC, rank and blood alcohol (Fig. 1, Extended Data Fig. 1, 5c), optogenetics with behaviour (Fig. 4, 6q, Extended Data Fig. 4), stability imaging (Extended Data Fig. 7), resident-intruder imaging (Extended Data Fig. 9), rank of imaged mice (Extended Data Fig. 6c,d) | blocked (see the attempt record) |

In [ ]:
import hashlib
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

import patel2026 as P

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
t_start = time.time()
print("dandiset", P.DANDISET_ID, "version", P.DANDISET_VERSION)

In [ ]:
A = P.assets()
print(len(A), "assets;", A.path.str.endswith(".nwb").sum(), "NWB files")
print(A.assign(kind=A.path.str.extract(r"_(icephys|behavior\+image\+ophys|image\+ophys)\.nwb$")[0].fillna("video"))
        .groupby("kind").agg(files=("path", "size"), GB=("size", lambda s: round(s.sum() / 1e9, 1))))
ic = A[A.path.str.endswith("icephys.nwb")].reset_index(drop=True)
t0 = time.time()
cells = P.parallel(P.load_cell, list(ic.asset_id), workers=32)
for c, p in zip(cells, ic.path):
    c["path"] = p
D = P.cell_table(cells)
print(f"loaded {len(cells)} neurons in {time.time() - t0:.0f} s")

## Data check

Each file holds one neuron. The electrode description says whether the neuron was a BLA-mPFC
projector or a nonspecific neighbour, the session description gives the housing, and the subject
description gives the rank (1 to 3) for most mice.

In [ ]:
print("electrode locations:", D.shape[0], "neurons,", pd.Series([c["location"] for c in cells]).value_counts().to_dict())
print("series descriptions:", pd.Series([c["series_description"] for c in cells]).value_counts().to_dict())
print("sweeps per file:", D.n_sweeps.value_counts().sort_index().to_dict())
g = D.groupby(["sex", "housing", "target"])
print(pd.DataFrame({"files": g.size(), "mice": g.subject.nunique(),
                    "recorded": g.recorded.sum(),
                    "mice_recorded": D[D.recorded].groupby(["sex", "housing", "target"]).subject.nunique()}))
# files that hold no recording
bad = D[~D.recorded]
print("\nfiles with no recording:", len(bad))
for i in bad.index:
    c = cells[i]
    print(f"  {c['path'].split('/')[-1]:70s} {D.housing[i]} V {c['V'].min():8.2f} to {c['V'].max():8.2f} mV, largest step {np.abs(c['step_pA']).max():5.1f} pA, sweeps {c['n_sweeps']}")

In [ ]:
# units and ranges of what is used, on the recorded neurons
R = D[D.recorded]
rc = [cells[i] for i in R.index]
err = [np.max(np.abs(c["step_pA"] - P.nominal_steps(c))[P.nominal_steps(c) != 0] / np.abs(P.nominal_steps(c)[P.nominal_steps(c) != 0])) for c in rc]
print("sampling rate 10 kHz, sweep 2 s: step onset sample", sorted({c["step_on"] for c in rc}), "offset", sorted({c["step_off"] for c in rc}),
      "-> step of", sorted({(c["step_off"] - c["step_on"]) / 10 for c in rc}), "ms (Methods: 500 ms)")
print(f"measured steps against -120 + 20 k pA: largest relative difference {max(err):.3f}")
print("first and last measured step (pA), range over neurons:",
      np.round(np.percentile([c["step_pA"][0] for c in rc], [0, 100]), 1),
      np.round(np.percentile([c["step_pA"][-1] for c in rc], [0, 100]), 1))
print("resting potential (mV), before the first step:", R.vrest.describe().round(1).to_dict())
print("voltage range over all sweeps (mV):", round(min(c["V"].min() for c in rc), 1), "to", round(max(c["V"].max() for c in rc), 1))
print("input resistance, last 100 ms (MOhm):", R.rin100.describe().round(1).to_dict())
for thr in (0, -20):
    E = P.curves(R, thr)
    print(f"spikes per 500-ms step, {thr} mV threshold: max {E.max():.0f}, median at 500 pA {np.median(E[:, -1]):.0f}")
# the same recording deposited twice?
h = pd.Series([hashlib.md5(c["V"].tobytes()).hexdigest() for c in cells])
print("files with identical voltage data:", int(h.duplicated(keep=False).sum()))
# step and response share one clock: the -120 pA response starts at the step onset
pre = np.array([c["V"][0, c["step_on"] - 100:c["step_on"]].mean() for c in rc])
post = np.array([c["V"][0, c["step_on"] + 100:c["step_on"] + 200].mean() for c in rc])
sh = np.array([c["V"][0, c["step_on"] - 300:c["step_on"] - 200].mean() for c in rc])
print(f"-120 pA: change from the 10 ms before onset to 10-20 ms after it {np.median(post - pre):.2f} mV; "
      f"the same contrast shifted 20 ms earlier {np.median(pre - sh):.2f} mV")

In [ ]:
# which neurons the paper's groups hold: neurons per mouse, by recording cohort (year in the subject id)
R = D[D.recorded]
t = R.groupby(["sex", "housing", "target", "cohort"]).agg(neurons=("path", "size"), mice=("subject", "nunique"))
print(t)
print("\nranks, recorded neurons (mice):")
print(R.groupby(["sex", "housing", "target", "rank"]).agg(neurons=("path", "size"), mice=("subject", "nunique")))

The groups against the paper's counts:

- Male BLA-mPFC, group-housed: the paper's Figure 2c has 21 neurons from 7 mice; the dandiset holds
  37 from 12. The 2022 cohort holds exactly 21 neurons from 7 mice, and the 16 neurons of the 2024
  cohort (5 subject ids) are not in Figure 2c. The rank analysis of Extended Data Fig. 3c (36
  neurons from 13 mice) uses both cohorts. The dandiset does not mark which neurons enter Figure 2c;
  the 2022 cohort is taken as Figure 2c's group (an assumption that rests on the counts), and every
  male result is also shown with all 37 neurons.
- Male BLA-mPFC, isolated: 30 neurons from 9 mice, as in the paper; one file holds a single sweep
  (-120 pA only), so 29 neurons enter the excitability tests.
- Female BLA-mPFC: 18 group-housed files from 9 mice (paper 18 from 9), of which 2 (one mouse) hold
  no recording; 14 isolated files from 6 mice (paper 15 from 7), of which 11 hold no recording, so 3
  neurons from 1 mouse remain. The claims that need isolated females are blocked.
- Nonspecific male neurons: 50 group-housed from 11 mice (paper 50 from 11) and 56 isolated from 11
  (paper 54 from 12).
- Ranks: male BLA-mPFC group-housed rank 1, 2, 3 hold 13, 7 and 17 neurons (paper 13, 9 and 14);
  nonspecific group-housed 10, 9 and 16 (paper 10, 9, 16); nonspecific isolated 17, 14 and 11
  (paper 16, 13, 11); female BLA-mPFC group-housed 5, 6 and 5 recorded (paper 7, 6, 5; the 2 empty
  files are rank 1).

In [ ]:
def anova_two(a, b, thr, labels=("a", "b")):
    """M6: two-way ANOVA, current (repeated) by group, neurons as units."""
    A, B = P.curves(a, thr), P.curves(b, thr)
    t = P.split_plot_anova(np.r_[A, B], pd.DataFrame({"grp": [labels[0]] * len(A) + [labels[1]] * len(B)}))
    ph = P.posthoc_two(A, B)
    return A, B, t, ph

def report_two(a, b, thr, name, labels):
    A, B, t, ph = anova_two(a, b, thr, labels)
    for lab, d, M in ((labels[0], a, A), (labels[1], b, B)):
        d = d[[x is not None for x in d[f"e{thr}"]]]
        print(f"  {lab} mean spikes per step by mouse:", d.assign(m=M.mean(1)).groupby("subject").m.mean().round(2).tolist())
    it, gr = t.loc["level:grp"], t.loc["grp"]
    print(f"{name}, {thr} mV: {labels[0]} n={len(A)} ({a.subject.nunique()} mice), {labels[1]} n={len(B)} ({b.subject.nunique()} mice)")
    print(f"  current x group F({it.df1:.0f}, {it.df2:.0f}) = {it.F:.2f}, p = {it.p:.2g}; group F({gr.df1:.0f}, {gr.df2:.0f}) = {gr.F:.2f}, p = {gr.p:.2g}")
    print(f"  steps where {labels[1]} > {labels[0]}: {(ph['diff'] > 0).sum()} of 26; "
          f"Sidak p < 0.05 at (per-step t) {list(ph.current_pA[ph.p_sidak_step < 0.05])} pA, (pooled error) {list(ph.current_pA[ph.p_sidak_pooled < 0.05])} pA; "
          f"smallest {ph.p_sidak_step.min():.3g} and {ph.p_sidak_pooled.min():.3g}")
    return A, B, t, ph

R = D[D.recorded]
mp = R[(R.sex == "M") & (R.target == "BLA-mPFC")]
fp = R[(R.sex == "F") & (R.target == "BLA-mPFC")]
ns = R[(R.sex == "M") & (R.target == "nonspecific")]
mGH22, mGHall, mSI = mp[(mp.housing == "GH") & (mp.cohort == "2022")], mp[mp.housing == "GH"], mp[mp.housing == "SI"]

## fig2c-male-blampfc-excitability-up

> After isolation BLA-mPFC neurons of males fire more action potentials across current injections
> than those of group-housed males.

Criterion: M6, labeled neurons of males; reproduced when the current by housing interaction has
p < 0.05, isolated neurons fire more at most steps, and at least one Sidak test across steps is
significant, under both spike readings (0 mV and -20 mV crossings). The group-housed group is the
2022 cohort (21 neurons, the paper's n); all 37 group-housed neurons are a variant.

In [ ]:
for thr in (0, -20):
    A, B, t, ph = report_two(mGH22, mSI, thr, "fig2c, group-housed 2022 cohort", ("GH", "SI"))
    if thr == 0:
        print(ph[["current_pA", "mean_a", "mean_b", "diff", "p_sidak_step", "p_sidak_pooled"]].round(3).to_string(index=False))
    report_two(mGHall, mSI, thr, "variant, all group-housed", ("GH", "SI"))
    print()
# per mouse: mean spikes over the 26 steps
for lab, d in [("GH 2022", mGH22), ("SI", mSI)]:
    m = d.assign(mean=[x.mean() for x in d.e0]).groupby("subject")["mean"].agg(["size", "mean"]).round(2)
    print(lab, "neurons and mean spikes per step (0 mV), by mouse:", m.to_dict("index"))

**Result: partially reproduced.** Under both spike readings the current by housing interaction
is significant (F(25, 1200) = 4.70 and 5.38, p < 0.0001), and isolated neurons fire more at 14 of
26 steps, from 60 to 320 pA (up to 2.6 more spikes, at 180 pA). They fire fewer spikes from
340 pA up, and no step differs after Sidak adjustment under either post hoc reading. With all 37
group-housed neurons the interaction holds (F = 4.65 and 5.69) and the per-step Sidak tests are
significant at 120 to 180 pA, and the pooled-error tests at 180 pA only. The interaction comes from a
steeper rise at low currents, not from more spikes across all steps.

## fig2e-female-higher-baseline-excitability

> Group-housed females have more excitable BLA-mPFC neurons than group-housed males.

Criterion: M6, group-housed labeled neurons; reproduced when the current by sex interaction has
p < 0.05, female neurons fire more at most steps, and at least one Sidak test is significant,
under both spike readings. Females: the 16 recorded group-housed neurons (paper 18).

In [ ]:
fGH = fp[fp.housing == "GH"]
for thr in (0, -20):
    A, B, t, ph = report_two(mGH22, fGH, thr, "fig2e, male 2022 cohort against female", ("M", "F"))
    print(ph[["current_pA", "mean_a", "mean_b", "diff", "p_sidak_step", "p_sidak_pooled"]].round(3).to_string(index=False))
    report_two(mGHall, fGH, thr, "variant, all group-housed males", ("M", "F"))
    print()

**Result: partially reproduced.** The current by sex interaction is strong under both readings
(F(25, 875) = 19.8 and 20.1, p < 0.0001). Female neurons fire more at low and middle currents (up
to 4.5 more spikes at 180 pA, Sidak significant at 140 to 180 pA under both post hoc readings) and
fewer at the highest currents (7.7 fewer at 500 pA by 0 mV crossings, also significant). With the
0 mV threshold females fire more at 16 of 26 steps; with -20 mV, at 12 of 26, which is not most
steps. So the criterion holds under one spike reading and not the other. The 37-male variant gives
the same pattern (19 and 13 of 26 steps).

## supt1-male-input-resistance-up

> Isolation raises the input resistance of male BLA-mPFC neurons.

Criterion: input resistance from the steady state of the -120 pA step, last 100 ms or last 50 ms;
unpaired t-test between housing groups, p < 0.05, under both readings. Group-housed neurons: the 2022
cohort as in Figure 2c (variant: all 37).

In [ ]:
for w in (100, 50):
    for lab, gh in [("GH 2022 cohort", mGH22), ("variant, all GH", mGHall)]:
        x, y = gh[f"rin{w}"].dropna(), mSI[f"rin{w}"].dropna()
        r = stats.ttest_ind(x, y)
        print(f"last {w} ms, {lab}: GH {x.mean():.1f} +- {x.sem():.1f} MOhm (n={len(x)}), SI {y.mean():.1f} +- {y.sem():.1f} (n={len(y)}), "
              f"difference {y.mean() - x.mean():.1f} MOhm, t({r.df:.0f}) = {r.statistic:.2f}, p = {r.pvalue:.3g}")
print("isolated neurons include the single-sweep file:", mSI.rin100.notna().sum(), "values")

**Result: not reproduced, resting on the choice of group-housed neurons.** With the 2022 cohort
(Figure 2c's 21 neurons), isolated neurons have higher input resistance (118.0 against 97.6 MOhm)
but the unpaired t-test gives p = 0.052 under both windows. With all 37 group-housed neurons
(90.6 MOhm) p = 0.002. Which neurons Supplementary Table 1 used is not stated, and its values are
not in the text read.

## ed2e-nonspecific-bla-excitability

> Isolation also increases the excitability of nonspecific BLA neurons in males, a small but
> significant interaction, which the Results call minimal.

Criterion: M6, unlabeled neighbouring neurons of males; reproduced when the interaction has p <
0.05 with isolated neurons firing more, under both spike readings. The F and the size of the
difference are reported beside those of fig2c.

In [ ]:
nGH, nSI = ns[ns.housing == "GH"], ns[ns.housing == "SI"]
for thr in (0, -20):
    A, B, t, ph = report_two(nGH, nSI, thr, "ed2e, nonspecific", ("GH", "SI"))
    print(f"  mean difference over steps {ph['diff'].mean():.2f} spikes, largest {ph['diff'].max():.2f} at {ph.current_pA[ph['diff'].idxmax()]} pA")
    A2, B2, t2, ph2 = anova_two(mGH22, mSI, thr)
    print(f"  beside fig2c: F = {t2.loc['level:grp', 'F']:.2f}, mean difference {ph2['diff'].mean():.2f}, largest {ph2['diff'].max():.2f}")

**Result: reproduced.** The current by housing interaction is significant for nonspecific
neurons (F(25, 2600) = 1.66, p = 0.021, at 0 mV; F = 1.87, p = 0.0057, at -20 mV), and isolated
neurons fire more at 17 of 26 steps (largest difference 1.3 spikes, at 240 pA), with no step significant after Sidak adjustment. The interaction F is smaller
than for BLA-mPFC neurons (4.70 and 5.38).

## ed2bcd-nonspecific-bla-no-rank

> Nonspecific BLA neurons of males show no rank differences in excitability, group-housed or
> isolated, and isolation does not change their firing at 300 pA within any rank.

Criterion: a claim of absence; b and c by M6 with rank (1 to 3) as the group factor, d by a
two-way ANOVA of rank by housing on spikes at 300 pA, each at p < 0.05. Reproduced when no rank
main effect or interaction reaches p < 0.05 in b and c, and no housing effect or interaction in d.
Neurons without a rank (the 2021 mice) are left out.

In [ ]:
def rank_anova(d, thr):
    d = d[d["rank"].notna()]
    Y = P.curves(d, thr)
    t = P.split_plot_anova(Y, pd.DataFrame({"rank": d["rank"].astype(int).astype(str).values}))
    return d, Y, t

for thr in (0, -20):
    for lab, d in [("b, group-housed", nGH), ("c, isolated", nSI)]:
        dd, Y, t = rank_anova(d, thr)
        means = {r: round(Y[dd["rank"].values == r].mean(), 2) for r in (1, 2, 3)}
        print(f"{lab}, {thr} mV: n per rank {dd['rank'].value_counts().sort_index().to_dict()}; "
              f"rank F({t.loc['rank', 'df1']:.0f}, {t.loc['rank', 'df2']:.0f}) = {t.loc['rank', 'F']:.2f}, p = {t.loc['rank', 'p']:.3f}; "
              f"current x rank F({t.loc['level:rank', 'df1']:.0f}, {t.loc['level:rank', 'df2']:.0f}) = {t.loc['level:rank', 'F']:.2f}, p = {t.loc['level:rank', 'p']:.3f}; mean spikes per step {means}")
    d = ns[ns["rank"].notna()]
    Y = P.curves(d, thr)
    k300 = int(np.nonzero(P.EXCITABILITY_CURRENTS == 300)[0][0])
    dd = pd.DataFrame({"y": Y[:, k300], "rank": d["rank"].astype(int).astype(str).values, "housing": d.housing.values})
    a = anova_lm(smf.ols("y ~ C(rank, Sum) * C(housing, Sum)", dd).fit(), typ=3)
    print(f"d, 300 pA, {thr} mV: housing F = {a.loc['C(housing, Sum)', 'F']:.2f}, p = {a.loc['C(housing, Sum)', 'PR(>F)']:.3f}; "
          f"rank x housing F = {a.loc['C(rank, Sum):C(housing, Sum)', 'F']:.2f}, p = {a.loc['C(rank, Sum):C(housing, Sum)', 'PR(>F)']:.3f}")
    print("   mean spikes at 300 pA:", dd.groupby(["rank", "housing"]).y.mean().round(2).to_dict())

**Result: reproduced.** No rank effect reaches p < 0.05 among group-housed (rank p = 0.26 and
0.25, interaction p = 0.23 and 0.20) or isolated nonspecific neurons (rank p = 0.48 and 0.34,
interaction p = 0.995 and 0.61). At 300 pA neither housing (p = 0.98) nor rank by housing (p =
0.13) is significant; the two spike readings give the same counts at 300 pA.

## ed3c-male-rank-blampfc-excitability

> Among group-housed males, dominant (rank 1) mice have less excitable BLA-mPFC neurons than
> intermediate (rank 2) and subordinate (rank 3) mice.

Criterion: M6 with rank (1 to 3) as the group factor, group-housed labeled male neurons (all 37,
both cohorts, as the paper's 36 neurons from 13 mice imply). Reproduced when the interaction has
p < 0.05, rank 1 fires least, and Tukey tests show rank 1 below both rank 2 and rank 3 at some
step; partially when only one of the two differs.

In [ ]:
for thr in (0, -20):
    dd, Y, t = rank_anova(mGHall, thr)
    rk = dd["rank"].values
    means = {r: round(Y[rk == r].mean(), 2) for r in (1, 2, 3)}
    print(f"{thr} mV: neurons (mice) per rank", {r: f"{(rk == r).sum()} ({dd[rk == r].subject.nunique()})" for r in (1, 2, 3)})
    print(f"  current x rank F({t.loc['level:rank', 'df1']:.0f}, {t.loc['level:rank', 'df2']:.0f}) = {t.loc['level:rank', 'F']:.2f}, p = {t.loc['level:rank', 'p']:.2g}; "
          f"rank F = {t.loc['rank', 'F']:.2f}, p = {t.loc['rank', 'p']:.3f}; mean spikes per step {means}")
    below = {2: [], 3: []}
    for j, cur in enumerate(P.EXCITABILITY_CURRENTS):
        groups = [Y[rk == r][:, j] for r in (1, 2, 3)]
        if np.all([g.std() == 0 for g in groups]):
            continue
        res = stats.tukey_hsd(*groups)
        for r in (2, 3):
            if res.pvalue[0, r - 1] < 0.05 and groups[r - 1].mean() > groups[0].mean():
                below[r].append(int(cur))
    print("  steps where Tukey puts rank 1 below rank 2:", below[2], "; below rank 3:", below[3])
    print("  rank means at 100, 200, 300, 400, 500 pA:", {r: Y[rk == r][:, [5, 10, 15, 20, 25]].mean(0).round(1).tolist() for r in (1, 2, 3)})

**Result: partially reproduced.** The current by rank interaction is significant (F(50, 850) =
3.46, p < 0.0001, under both readings), and rank 1 neurons fire least (5.57 spikes per step
against 7.43 and 7.40). Tukey tests put rank 1 below rank 3 at 360 to 500 pA, but below rank 2 at
no step. Rank 2 holds 7 neurons from 3 mice here, against 9 from 4 in the paper.

## ed3d-female-rank-blampfc-none

> Group-housed females show no rank differences in BLA-mPFC excitability.

Criterion: a claim of absence; M6 with rank as the group factor at p < 0.05. Reproduced when
neither the rank main effect nor the interaction reaches p < 0.05.

In [ ]:
for thr in (0, -20):
    dd, Y, t = rank_anova(fGH, thr)
    rk = dd["rank"].values
    print(f"{thr} mV: neurons (mice) per rank", {r: f"{(rk == r).sum()} ({dd[rk == r].subject.nunique()})" for r in (1, 2, 3)},
          f"rank F({t.loc['rank', 'df1']:.0f}, {t.loc['rank', 'df2']:.0f}) = {t.loc['rank', 'F']:.2f}, p = {t.loc['rank', 'p']:.3f}; "
          f"current x rank F({t.loc['level:rank', 'df1']:.0f}, {t.loc['level:rank', 'df2']:.0f}) = {t.loc['level:rank', 'F']:.2f}, p = {t.loc['level:rank', 'p']:.3f}; "
          "mean spikes per step", {r: round(Y[rk == r].mean(), 2) for r in (1, 2, 3)})

**Result: reproduced.** Neither rank (p = 0.43 and 0.83) nor current by rank (p > 0.99) reaches
p < 0.05 among the 16 recorded group-housed female neurons; rank 1 has 5 neurons from 2 mice, as
2 of the paper's 7 are in files without a recording.

## Blocked: isolated females (fig2d, fig2-threeway, supt1-female)

These claims need isolated female BLA-mPFC neurons. Of the 14 files, 11 (5 mice) hold no
recording, so only 3 neurons from one mouse remain (paper 15 from 7). The numbers below are shown
for completeness and decide no outcome.

In [ ]:
fSI = fp[fp.housing == "SI"]
nall = int(((D.sex == "F") & (D.housing == "SI")).sum())
print(f"isolated female files: {nall}, recorded {len(fSI)}, mice with a recording {fSI.subject.unique().tolist()}")
for thr in (0, -20):
    A, B, t, ph = anova_two(fGH, fSI, thr)
    print(f"{thr} mV: 16 GH against {len(B)} SI neurons, interaction F = {t.loc['level:grp', 'F']:.2f}, p = {t.loc['level:grp', 'p']:.3f}; SI below GH at {(ph['diff'] < 0).sum()} of 26 steps")

## Summary

| Claim | Outcome |
|---|---|
| fig2c-male-blampfc-excitability-up | partially reproduced (assumed: Figure 2c's group-housed neurons are the 2022 cohort) |
| fig2e-female-higher-baseline-excitability | partially reproduced |
| supt1-male-input-resistance-up | not reproduced (assumed, as fig2c; p = 0.052) |
| ed2e-nonspecific-bla-excitability | reproduced |
| ed2bcd-nonspecific-bla-no-rank | reproduced |
| ed3c-male-rank-blampfc-excitability | partially reproduced |
| ed3d-female-rank-blampfc-none | reproduced |
| fig2d, fig2-threeway, supt1-female | blocked: 11 of 14 isolated female files hold no recording |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: spikes are counted as threshold crossings (0 or -20 mV,
with a 5 mV re-arm), not in Clampfit; the post hoc tests are Sidak-adjusted t-tests at each step,
with the step's own variance or the error pooled over steps, not Prism's; the ANOVA is the
sphericity-assumed univariate split-plot analysis; Figure 2c's group-housed males are taken to be
the 2022 cohort, which matches the paper's 21 neurons from 7 mice.

What was found about the dataset: 13 of the 205 files (11 isolated and 2 group-housed female
BLA-mPFC neurons, 7 mice) hold no recording (flat voltage at about 0 or -209 mV and no step in the
stimulus); one isolated male file holds only the -120 pA sweep; nothing marks which group-housed male
neurons form Figure 2c; neuron counts per rank differ from the paper's in some groups (male
BLA-mPFC rank 2 and 3, nonspecific isolated); 15 nonspecific neurons (2021 mice) carry no rank.

In [ ]:
print(f"runtime {(time.time() - t_start) / 60:.1f} min")